In [ ]:
-- The correct usage is: EXPLAIN <single SELECT/INSERT/UPDATE/DELETE/MERGE statement>
-- To EXPLAIN a CTE, wrap the CTE and the final SELECT in parentheses, or just EXPLAIN the SELECT using the CTE.
-- The following is the corrected code for the performance test section (section 11):

EXPLAIN
SELECT
    -- Source columns
    CAST(product AS STRING) AS product,
    CAST(ship_date AS DATE) AS ship_date,
    CAST(days_supply AS STRING) AS days_supply,
    CAST(qty AS STRING) AS qty,
    CAST(treatment_id AS STRING) AS treatment_id,
    CAST(dob AS DATE) AS dob,
    CAST(first_ship_date AS DATE) AS first_ship_date,
    CAST(refill_status AS STRING) AS refill_status,
    CAST(patient_id AS STRING) AS patient_id,
    CAST(ship_type AS STRING) AS ship_type,
    CAST(shipment_arrived_status AS STRING) AS shipment_arrived_status,
    CAST(delivery_ontime AS STRING) AS delivery_ontime,
    -- Derived columns
    DATE_ADD(
      ship_date,
      CAST(
        COALESCE(
          CAST(days_supply AS INT),
          CAST(qty AS INT) / 3 * 7
        ) AS INT
      )
    ) AS shipment_expiry,
    DATE_ADD(
      DATE_ADD(
        ship_date,
        CAST(
          COALESCE(
            CAST(days_supply AS INT),
            CAST(qty AS INT) / 3 * 7
          ) AS INT
        )
      ),
      91
    ) AS discontinuation_date,
    DATEDIFF(
      DATE_ADD(
        ship_date,
        CAST(
          COALESCE(
            CAST(days_supply AS INT),
            CAST(qty AS INT) / 3 * 7
          ) AS INT
        )
      ),
      calctime
    ) + 1 AS days_until_next_ship,
    DATEDIFF(calctime, ship_date) + 1 AS days_since_last_fill,
    DATE_ADD(calctime, DATEDIFF(
      DATE_ADD(
        ship_date,
        CAST(
          COALESCE(
            CAST(days_supply AS INT),
            CAST(qty AS INT) / 3 * 7
          ) AS INT
        )
      ),
      calctime
    ) + 1) AS expected_refill_date,
    LAG(ship_date) OVER(PARTITION BY treatment_id ORDER BY ship_date) AS prior_ship,
    CASE
      WHEN LAG(ship_date) OVER(PARTITION BY treatment_id ORDER BY ship_date) IS NOT NULL
      THEN DATEDIFF(ship_date, LAG(ship_date) OVER(PARTITION BY treatment_id ORDER BY ship_date))
      ELSE NULL
    END AS days_between,
    CASE
      WHEN DATEDIFF(
        calctime,
        DATE_ADD(
          ship_date,
          CAST(
            COALESCE(
              CAST(days_supply AS INT),
              CAST(qty AS INT) / 3 * 7
            ) AS INT
          )
        )
      ) >= 0
      THEN DATEDIFF(
        calctime,
        DATE_ADD(
          ship_date,
          CAST(
            COALESCE(
              CAST(days_supply AS INT),
              CAST(qty AS INT) / 3 * 7
            ) AS INT
          )
        )
      )
      ELSE NULL
    END AS days_since_supply_out,
    FLOOR(DATEDIFF(dob, calctime) / 365.25) AS age,
    ROUND(DATEDIFF(dob, first_ship_date) / 365.0, 0) AS age_at_first_ship,
    COUNT(
      CASE WHEN ship_type = "commercial" THEN ship_date END
    ) OVER(PARTITION BY patient_id, treatment_id) AS latest_therapy_ships,
    CASE
      WHEN refill_status = "DC - Standard" THEN "STANDARD"
      WHEN refill_status = "DC-PERMANENT" THEN "PERMANENT"
      ELSE NULL
    END AS discontinuation_type
FROM purgo_playground.patient_therapy_shipment;
